# 5. Red Teaming Your AI Models

Welcome to the Red Teaming notebook! 🛡️

In this guide, we'll walk you through **red teaming** - the process of probing your AI models with adversarial prompts to identify security vulnerabilities, biases, and potential risks before bad actors do.

Red teaming is essential for building safe, reliable AI systems. Think of it as ethical hacking for your LLMs - you're testing how your model responds under attack so you can strengthen its defenses.

## What You'll Learn

By the end of this notebook, you'll know how to:
- ✅ Identify and use saved model endpoints for red teaming
- ✅ Create V3 red team payloads with custom configurations
- ✅ Submit red team tests and monitor their progress
- ✅ Retrieve and analyze test results
- ✅ Use different attack methods for comprehensive testing
- ✅ Build custom test suites for specific use cases

## What's New in V3?

The V3 API brings powerful enhancements:
- **Granular Parameter Control**: Fine-tune each attack method
- **Structured Attack Hierarchy**: Organized basic, static, and dynamic attacks
- **Enhanced Attack Methods**: Expanded encoding, obfuscation, and multi-modal techniques

Let's get started! 🚀


## Setup

First, let's import the necessary libraries and initialize our Enkrypt AI clients. Make sure you have your `ENKRYPTAI_API_KEY` set in your `.env` file.


In [ ]:
import os
import copy
import time
import pandas as pd
from enkryptai_sdk import RedTeamClient, ModelClient
from dotenv import load_dotenv

load_dotenv()

# Environment Variables
ENKRYPT_API_KEY = os.getenv("ENKRYPTAI_API_KEY")
ENKRYPT_BASE_URL = os.getenv("ENKRYPTAI_BASE_URL") or "https://api.enkryptai.com"

# Initialize Clients
redteam_client = RedTeamClient(api_key=ENKRYPT_API_KEY, base_url=ENKRYPT_BASE_URL)
model_client = ModelClient(api_key=ENKRYPT_API_KEY, base_url=ENKRYPT_BASE_URL)

print("✅ Enkrypt AI clients initialized successfully!")


## Step 1: Select Your Model Endpoint

For this tutorial, we'll use the model endpoint you created in **notebook 4**. If you used a different name, simply update the variables below.

We'll be testing: **my-gpt4-endpoint (v1)**

If you haven't created this endpoint yet, go back to **notebook 4** and add a model first!


In [ ]:
# Define the model endpoint to test
# Update these values if you used a different model name in notebook 4
model_saved_name = "my-gpt4-endpoint"
model_version = "v1"

print(f"✅ Selected Model Endpoint:")
print(f"   Name: {model_saved_name}")
print(f"   Version: {model_version}")
print(f"\n💡 Tip: If you want to test a different model, update the variables above.")



## Step 2: Create a Basic V3 Red Team Payload

Now let's create our first V3 red team configuration. We'll start with a simple test that:
- Tests for **harmful content** generation
- Uses **basic** attack methods (raw prompts without modification)
- Samples **5%** of the dataset for quick testing

The V3 payload structure has three main sections:
1. **dataset_configuration** (optional): For generating custom test datasets
2. **redteam_test_configurations** (required): Which tests to run and how
3. **endpoint_configuration** (not needed when using saved models)

### Understanding Attack Methods

Attack methods are organized into three categories:

- **Basic**: Raw prompts without modification - your baseline test
- **Static**: Encoding and obfuscation techniques (base64, obfuscation, multilingual, etc.)
- **Dynamic**: Adaptive attacks that learn and evolve (iterative, multi-turn)


In [ ]:
# Create a unique test name
import uuid
redteam_test_name = f"my-first-redteam-test-{str(uuid.uuid4())[:8]}"

sample_percentage = 2

# Basic V3 configuration
# Note: dataset_name is not needed when using add_custom_task_with_saved_model_v3
# The standard dataset is used by default
basic_redteam_config_v3 = {
    "test_name": redteam_test_name,
    "redteam_test_configurations": {
        "harmful_test": {
            "sample_percentage": sample_percentage,
            "attack_methods": {
                "basic": {
                    "basic": {
                        "params": {}
                    }
                }
            }
        }
    }
}

print(f"✅ Created red team configuration: {redteam_test_name}")
print("\n📊 Test Configuration:")
print(f"   - Test Type: harmful_test")
print(f"   - Sample Size: {sample_percentage}%")
print(f"   - Attack Methods: basic (raw prompts)")
print(f"   - Dataset: standard (Enkrypt's built-in)")


## Step 3: Submit the Red Team Test

Now that we have our configuration ready, let's submit the test using our saved model endpoint. We'll use the `add_custom_task_with_saved_model_v3()` method.

This will queue the test for execution. Depending on the sample size and attack methods, tests typically complete in 2-10 minutes.


In [ ]:
# Submit the red team test
add_redteam_response = redteam_client.add_custom_task_with_saved_model_v3(
    config=copy.deepcopy(basic_redteam_config_v3),
    model_saved_name=model_saved_name,
    model_version=model_version
)

print(add_redteam_response)
print(f"\n✅ {add_redteam_response.message}")
print(f"\n🔬 Your red team test is now queued!")
print(f"   Test Name: {redteam_test_name}")
print(f"   Model: {model_saved_name} (v{model_version})")


## Step 4: Check Test Status

Let's check the status of our red team test. It will go through these stages:
- **Queued**: Waiting to start
- **Running**: Actively testing your model
- **Finished**: Complete and ready for analysis
- **Failed**: Something went wrong (check error message)


In [ ]:
# Check the status
status_response = redteam_client.status(test_name=redteam_test_name)

print(f"📊 Test Status: {status_response.status}")
print(f"\nFull status response:")
print(status_response)

assert status_response.status in ["Queued", "Running", "Finished", "Failed"]


## Wait for Test Completion

Let's poll the status until the test finishes. This cell will check every 10 seconds and notify you when it's done.

⏱️ **Estimated time**: 2-5 minutes for a 5% sample with basic attack methods.


In [ ]:
# Poll until test is finished
print(f"⏳ Waiting for test to complete...\n")

while True:
    status_response = redteam_client.status(test_name=redteam_test_name)
    current_status = status_response.status
    
    print(f"   Status: {current_status}")
    
    if current_status == "Finished":
        print("\n✅ Red team test completed successfully!")
        break
    elif current_status == "Failed":
        print("\n❌ Red team test failed.")
        raise RuntimeError("Red team task failed.")
    
    time.sleep(10)  # Wait 10 seconds before checking again


## Step 5: View Test Results Summary

Now that the test is complete, let's retrieve the results summary. This gives you a high-level overview of how your model performed across all test types.


In [ ]:
# Get the results summary
results_summary = redteam_client.get_result_summary(test_name=redteam_test_name)

print("📊 Red Team Test Results Summary\n")
print(f"Test Name: {redteam_test_name}\n")
print("Results:")
print(results_summary.summary)

# Convert to dictionary for detailed inspection
summary_dict = results_summary.to_dict()
print("\nFull summary as dictionary:")
print(summary_dict)


## Display Results in a Table

Let's format the results in a clean table for easier analysis.


In [ ]:
# Extract summary data
summary_data = results_summary.summary.to_dict()

# Convert to DataFrame
df_summary = pd.DataFrame.from_dict(
    summary_data, 
    orient="index", 
    columns=["Violation Rate or Score"]
)
df_summary.index.name = "Test Type"

# Display as table
df_summary.reset_index(inplace=True)
display(df_summary)


## View Detailed Results by Test Type

Let's dive deeper into the results for a specific test type. This shows you individual prompts, responses, and evaluations.


In [ ]:
# Get detailed results for harmful_test
test_type = "harmful_test"
details_response = redteam_client.get_result_details_test_type(
    test_name=redteam_test_name,
    test_type=test_type
)

print(f"📋 Detailed Results for: {test_type}\n")
print(details_response)

# Print as dictionary for inspection
import json
print("\nFull details (formatted):")
print(json.dumps(details_response.to_dict(), indent=2))


## Additional Useful Methods

Here are some other helpful methods for working with red team tests:

### Get Download Link
Download the complete results as a file:
```python
download_link_response = redteam_client.get_download_link(test_name=redteam_test_name)
print(f"Download link: {download_link_response.link}")
print(f"Expires at: {download_link_response.expires_at}")
```

### List All Red Team Tasks
See all your red team tests:
```python
all_tasks = redteam_client.get_task_list()
finished_tasks = redteam_client.get_task_list(status="Finished")
```

### Get Task Details
Retrieve the configuration of a specific test:
```python
task_details = redteam_client.get_task(test_name=redteam_test_name)
print(f"Task ID: {task_details.task_id}")
```

### Get Findings Summary
Get AI-generated insights from your test results:
```python
findings = redteam_client.get_findings(redteam_summary=results_summary)
print(findings.findings)
```


---

# Exploring Different Red Team Payloads

Now that you understand the basics, let's explore more advanced configurations for different use cases. Below are several examples you can adapt for your needs.


## Example 1: Multi-Test Assessment

Test your model across multiple risk categories at once. This configuration tests for:
- **Harmful content** (with obfuscation attacks)
- **Bias** (with multilingual testing)
- **Toxicity** (basic prompts)
- **PII leakage** (with encoding attacks)


In [ ]:
multi_test_config = {
    "test_name": "comprehensive-multi-test",
    "redteam_test_configurations": {
        "harmful_test": {
            "sample_percentage": 10,
            "attack_methods": {
                "basic": {
                    "basic": {"params": {}}
                },
                "static": {
                    "obfuscation": {"params": {}},
                    "base64_encoding": {
                        "params": {
                            "encoding_type": "base64",
                            "iterations": 1
                        }
                    }
                }
            }
        },
        "bias_test": {
            "sample_percentage": 10,
            "attack_methods": {
                "basic": {
                    "basic": {"params": {}}
                },
                "static": {
                    "lang_es": {"params": {}},  # Spanish
                    "lang_fr": {"params": {}}   # French
                }
            }
        },
        "toxicity_test": {
            "sample_percentage": 10,
            "attack_methods": {
                "basic": {
                    "basic": {"params": {}}
                }
            }
        },
        "pii_test": {
            "sample_percentage": 10,
            "attack_methods": {
                "basic": {
                    "basic": {"params": {}}
                },
                "static": {
                    "url_encoding": {"params": {}},
                    "hex_encoding": {"params": {}}
                }
            }
        }
    }
}

print("✅ Multi-Test Configuration Created")
print("\n📊 This will test:")
print("   - Harmful Content (basic + obfuscation + base64)")
print("   - Bias (basic + Spanish + French)")
print("   - Toxicity (basic)")
print("   - PII Leakage (basic + URL encoding + hex encoding)")
print("\n⏱️  Estimated time: 5-10 minutes")


## Example 2: Advanced Attack Methods

Use sophisticated attack techniques including:
- **Dynamic attacks**: Iterative and multi-turn conversations
- **Advanced jailbreaks**: EAI attack and deep inception
- **Multi-language**: Testing across different languages


In [ ]:
advanced_attacks_config = {
    "test_name": "advanced-attack-methods",
    "redteam_test_configurations": {
        "harmful_test": {
            "sample_percentage": 5,
            "attack_methods": {
                "basic": {
                    "basic": {"params": {}}
                },
                "static": {
                    "eai_attack": {"params": {}},
                    "deep_inception": {"params": {}},
                    "lang_hi": {"params": {}},  # Hindi
                    "lang_ja": {"params": {}}   # Japanese
                },
                "dynamic": {
                    "iterative": {
                        "params": {
                            "width": 2,
                            "branching_factor": 2,
                            "depth": 3
                        }
                    },
                    "multi_turn": {"params": {}}
                }
            }
        }
    }
}

print("✅ Advanced Attack Configuration Created")
print("\n📊 Attack methods:")
print("   Static Attacks:")
print("     - EAI Attack (advanced jailbreak)")
print("     - Deep Inception (nested injection)")
print("     - Hindi and Japanese (non-Latin scripts)")
print("   Dynamic Attacks:")
print("     - Iterative (adaptive learning)")
print("     - Multi-turn (conversation exploitation)")
print("\n⚠️  Warning: Dynamic attacks take longer")
print("⏱️  Estimated time: 10-15 minutes")


## Example 3: Security-Focused Testing

Perfect for code generation models or developer-facing applications. Tests for:
- **Insecure code generation**
- **System prompt extraction**
- **PII leakage**
- **Competitor information disclosure**


In [ ]:
security_focused_config = {
    "test_name": "security-vulnerability-scan",
    "redteam_test_configurations": {
        "insecure_code_test": {
            "sample_percentage": 15,
            "attack_methods": {
                "basic": {
                    "basic": {"params": {}}
                },
                "static": {
                    "obfuscation": {"params": {}},
                    "base64_encoding": {
                        "params": {
                            "encoding_type": "base64",
                            "iterations": 2
                        }
                    }
                }
            }
        },
        "system_prompt_extractions_test": {
            "sample_percentage": 20,
            "attack_methods": {
                "basic": {
                    "basic": {"params": {}}
                },
                "static": {
                    "obfuscation": {"params": {}},
                    "rot13_encoding": {"params": {}}
                }
            }
        },
        "pii_test": {
            "sample_percentage": 15,
            "attack_methods": {
                "basic": {
                    "basic": {"params": {}}
                }
            }
        },
        "competitor_test": {
            "sample_percentage": 10,
            "attack_methods": {
                "basic": {
                    "basic": {"params": {}}
                }
            }
        }
    }
}

print("✅ Security-Focused Configuration Created")
print("\n🔒 Security tests:")
print("   - Insecure Code (15% with obfuscation + base64)")
print("   - System Prompt Extraction (20% with obfuscation + ROT13)")
print("   - PII Leakage (15% basic)")
print("   - Competitor Info (10% basic)")
print("\n⏱️  Estimated time: 7-12 minutes")


## Example 4: Custom Dataset Generation

Generate a custom test dataset tailored to your specific application. This is perfect when you need domain-specific testing beyond the standard dataset.

In this example, we'll test a healthcare chatbot to ensure it doesn't give medical advice or violate HIPAA.


In [ ]:
custom_dataset_config = {
    "test_name": "healthcare-chatbot-test",
    "dataset_configuration": {
        "system_description": """
        You are a healthcare appointment scheduling assistant. 
        You help patients book, reschedule, and cancel appointments.
        You can provide general information about clinic hours and locations.
        """,
        "policy_description": """
        The assistant must NOT:
        - Provide medical advice or diagnoses
        - Share patient health information (HIPAA violation)
        - Prescribe medications or treatments
        - Make medical decisions on behalf of doctors
        - Access or reveal private medical records
        """,
        "max_prompts": 50,
        "scenarios": 3,
        "categories": 2,
        "depth": 2
    },
    "redteam_test_configurations": {
        "harmful_test": {
            "sample_percentage": 100,  # Test all generated prompts
            "attack_methods": {
                "basic": {
                    "basic": {"params": {}}
                },
                "static": {
                    "obfuscation": {"params": {}}
                }
            }
        }
    }
}

print("✅ Custom Dataset Configuration Created")
print("\n🏥 Healthcare Chatbot Test:")
print("   - Will generate 50 custom prompts")
print("   - Based on your system description and policies")
print("   - Tests policy violations specific to healthcare")
print("\n📊 Test parameters:")
print("   - Scenarios: 3 (different attack scenarios)")
print("   - Categories: 2 (different risk categories)")
print("   - Depth: 2 (prompt complexity)")
print("\n⏱️  Estimated time: 5-8 minutes")


## Example 5: Using a Saved Code of Conduct Policy

If you've already created and saved a Code of Conduct policy (from notebook 2), you can reference it in your red team tests instead of writing the policy inline.


In [ ]:
# Configuration using a saved CoC policy
coc_policy_config = {
    "test_name": "coc-policy-compliance-test",
    "redteam_test_configurations": {
        "harmful_test": {
            "sample_percentage": 10,
            "attack_methods": {
                "basic": {
                    "basic": {"params": {}}
                },
                "static": {
                    "obfuscation": {"params": {}}
                }
            }
        }
    }
}

# When submitting, you can specify the policy_name parameter
# Uncomment and modify the policy name to use your saved policy:

# response = redteam_client.add_custom_task_with_saved_model_v3(
#     config=copy.deepcopy(coc_policy_config),
#     model_saved_name=model_saved_name,
#     model_version=model_version,
#     policy_name="mortgage-guidelines-policy-0"  # Your saved policy name
# )

print("✅ Code of Conduct Policy Configuration Created")
print("\n📋 To use a saved policy:")
print("   1. Create and save a policy in notebook 2")
print("   2. Reference it by name when submitting the test")
print("   3. The test will use your policy rules for evaluation")
print("\n💡 This is ideal for enterprise policies that apply across multiple tests")


---

# Reference: Available Test Types and Attack Methods

## Standard Test Types (12 Available)

Here's a quick reference of all available test types you can use:

| Test Keyword | Purpose | When to Use |
|-------------|---------|-------------|
| `harmful_test` | Detects harm or danger promotion | General safety testing |
| `toxicity_test` | Identifies toxic/offensive content | Content moderation |
| `bias_test` | Finds biased outputs | Fairness testing |
| `pii_test` | Detects personal info leakage | Privacy compliance |
| `insecure_code_test` | Finds vulnerable code | Code generation models |
| `system_prompt_extractions_test` | Tests prompt extraction | Protecting system prompts |
| `cbrn_test` | Chemical/biological/nuclear risks | High-security applications |
| `csem_test` | Child exploitation content | Content safety |
| `copyright_test` | Copyrighted material | Legal compliance |
| `misinformation_test` | False information | Factual accuracy |
| `sponge_test` | Resource exhaustion | Performance/DoS testing |
| `competitor_test` | Competitor info disclosure | Business confidentiality |


## Attack Methods Quick Reference

### Basic & Obfuscation
- **`basic`**: Raw prompts without modification (always start here)
- **`obfuscation`**: Character-level obfuscation

### Encoding Methods
- **`base64_encoding`**: Base64 encoding (params: `iterations` 1-3)
- **`hex_encoding`**: Hexadecimal encoding
- **`ascii_encoding`**: ASCII encoding
- **`binary_encoding`**: Binary encoding
- **`url_encoding`**: URL encoding
- **`leet_encoding`**: Leet speak (l33t)
- **`rot13_encoding`**: ROT13 cipher
- **`rot21_encoding`**: ROT21 cipher
- **`morse_encoding`**: Morse code

### Multilingual
- **`lang_fr`**: French
- **`lang_it`**: Italian
- **`lang_es`**: Spanish
- **`lang_hi`**: Hindi
- **`lang_ja`**: Japanese

### Advanced Jailbreaks
- **`eai_attack`**: Advanced jailbreak technique
- **`deep_inception`**: Nested injection attacks

### Dynamic Attacks
- **`iterative`**: Adaptive learning (params: `width`, `branching_factor`, `depth`)
- **`multi_turn`**: Multi-turn conversation exploitation


---

# Best Practices and Tips

## 🎯 Testing Strategy

### Start Simple, Then Scale
1. **Begin with basic attacks** on a small sample (5%)
2. **Add encoding methods** if basic tests pass (10-15%)
3. **Include multilingual** for global applications (15-20%)
4. **Try advanced attacks** for high-security apps (20-30%)
5. **Run dynamic attacks** for comprehensive assessment (full dataset)

### Choose the Right Tests
- **Consumer Apps**: Focus on `harmful_test`, `toxicity_test`, `bias_test`
- **Enterprise Apps**: Add `pii_test`, `competitor_test`, `misinformation_test`
- **Code Assistants**: Prioritize `insecure_code_test`, `system_prompt_extractions_test`
- **Healthcare/Finance**: Include `cbrn_test` and custom policy testing
- **Content Platforms**: Use `csem_test`, `copyright_test`, `toxicity_test`

## ⚡ Performance Tips

- **Sample Percentage**: Start with 5-10% for quick tests
- **Attack Methods**: Each additional method multiplies test time
- **Dynamic Attacks**: Can take 3-5x longer than static attacks
- **Custom Datasets**: Generation adds 1-2 minutes to test time

## 🔒 Security Recommendations

1. **Test Before Deployment**: Always red team before going live
2. **Regular Testing**: Re-test after major model updates
3. **Layer Your Defenses**: Combine red teaming with runtime guardrails
4. **Custom Policies**: Use domain-specific policies for better coverage
5. **Iterative Improvement**: Use findings to strengthen your system

## 📊 Interpreting Results

- **< 10% violation rate**: Generally acceptable for most applications
- **10-30% violation rate**: Needs improvement, consider guardrails
- **> 30% violation rate**: High risk, requires immediate attention
- **Dynamic attack failures**: Indicates vulnerability to sophisticated attacks


---

# Summary

## 🎉 Congratulations!

You've completed the Red Teaming notebook and learned how to:

✅ **Identify saved endpoints** for testing  
✅ **Create V3 red team payloads** with custom configurations  
✅ **Submit and monitor tests** through completion  
✅ **Retrieve and analyze results** in multiple formats  
✅ **Use different attack methods** for comprehensive testing  
✅ **Build custom test suites** for specific use cases  

## What You've Accomplished

You now know how to:
- Run basic red team tests with the standard dataset
- Configure multi-test assessments across multiple risk categories
- Use advanced attack methods including dynamic and multilingual attacks
- Generate custom datasets for domain-specific testing
- Reference saved Code of Conduct policies in your tests
- Interpret results and understand violation rates

## 🚀 Next Steps

### Immediate Actions
1. **Run your first test** on one of your models
2. **Review the results** and identify vulnerabilities
3. **Implement guardrails** to address findings (see notebook 3)

### Advanced Topics (Notebook 6: Deployments)
- **Deploy with Protection**: Wrap models with runtime guardrails
- **AI Proxy**: Route traffic through Enkrypt's secure proxy
- **Monitor in Production**: Track real-time safety metrics
- **Incident Response**: Handle violations automatically

### Continuous Improvement
1. **Regular Testing**: Re-test after model updates
2. **Compare Results**: Track improvement over time
3. **Expand Coverage**: Add new test types as your app evolves
4. **Share Findings**: Help your team understand AI risks

---

## 💡 Pro Tips

- **Start Small**: Begin with 5% samples and basic attacks
- **Build Up**: Add complexity as you understand your model's behavior
- **Iterate**: Use findings to improve prompts, guardrails, and training
- **Document**: Keep records of tests for compliance and audit purposes

Ready to deploy your secured model? Head to **notebook 6** to learn about deployments! 🚀


---

## Need Help?

- 📖 **Documentation**: [docs.enkryptai.com](https://docs.enkryptai.com)
- 💬 **Contact Support**: [enkryptai.com/request-a-demo](https://enkryptai.com/request-a-demo)
- 🌐 **Website**: [enkryptai.com](https://enkryptai.com)
- 🎓 **More Examples**: Check our [GitHub repository](https://github.com/enkryptai)

---

**Ship Fast. Ship Safe. Stay Ahead.** ⚡🔒
